# Fleet Maintenance Planning: final report

**Competition:** Kaggle `mlprojectfleet2`, Problem 2 (CSE 343 / ECE 363). **Team:** Group 16. **Repository:** `Daksh183/fleet-maintenance`.

For each of 20,000 vehicles from a **new depot**, decide whether to **flag** it for inspection and what spare-parts **budget** to set,
so that the total cost is as low as possible.

**The three selected models** (the three submissions selected on Kaggle). Running this notebook top to bottom regenerates their CSVs
in `outputs/` and checks each one against the uploaded file.

| Model | Version | Idea | Kaggle file | Public score |
|---|---|---|---|---|
| **Model 1** | v3 | XGBoost failure probability with a flag rule and newsvendor budget derived from the cost function | `submission_v3.csv` | 27,760.14 |
| **Model 2** | v34 | v12-style flags with Gaussian-process repair budgets that use each vehicle's own uncertainty and a context-corrected failure probability | `submission_v34_budget.csv` | 16,587.74 |
| **Model 3** | v51 | Bayes decision under a posterior over test-depot hypotheses, with a reduced-input Gaussian-process repair-cost model (**best public score**) | `submission_v51_gp_reduced.csv` | **16,388.51** |

Public scores are total cost on the public 30% of test (about 6,000 vehicles); lower is better. The baseline (v2: plain XGBoost with a
0.5 threshold) scored 74,933 on the same board. **Leaderboard: rank 13.** The final upload, v82, scored 16,440.10; our best public
score is 16,388.51 (Model 3).

**How this notebook maps to the evaluation scheme (instructions, part 2)**

| Criterion | Where |
|---|---|
| (a) Leaderboard ranking | Sections 0 and 9 |
| (b) Performance gains across the three model versions | Sections 5–8 |
| (c) Data and error analyses behind the improvements; limitations and capabilities of the best model | Sections 2, 3, 4 and 9 |
| (d) Individual contributions | Section 10 |
| (e) Understanding of the models, and the reasoning behind the choice of models and features | Sections 1, 4 and 5–7 (each model is explained before its code) |

**To run:** put `train.csv` and `test.csv` from the competition in `../data/` and run from the `notebooks/` folder. Models 2 and 3
load saved artifacts from `../reports/` (probabilities, out-of-fold predictions, public-score posteriors), as the course asked on 26 Nov
("save your trained artifacts"); each artifact is listed with the script in `../src/` that produced it. Libraries: numpy, pandas,
scipy, scikit-learn, xgboost.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

## 1. The cost function and the two decisions it implies

For a vehicle with repair cost $c$ ($c = 0$ if it does not fail):

- not flagged: cost $= c$
- flagged with budget $b$: cost $= 10 + 0.20\,(c-b)^+ + 0.02\,(b-c)^+$

**Budget (newsvendor).** A unit of shortfall costs 0.20 and a unit of unused budget 0.02, so the best budget is the
$q^* = 0.20/0.22 = 0.909$ quantile of the vehicle's cost distribution. A vehicle fails with probability $p$, so its cost is 0 with
probability $1-p$, and
$b^* = 0$ if $p \le 0.0909$, else $b^* = F^{-1}\!\left(1 - 0.0909/p\right)$, where $F$ is the repair-cost distribution given failure.

**Flag.** Flag when the expected cost of flagging (with $b^*$) is below the expected cost of not flagging, $p\,\mu$. With $b^* = 0$
this means $p\,\mu > 12.5$; with the mean training repair cost $\mu \approx 492$, **flag when $p \gtrsim 2.5\%$**, far below 0.5.

So every model needs two things per vehicle: a **failure probability** $p$ and a **repair-cost distribution** $F$. Each model below
improves one or both, or the way the decision combines them.

## 2. Data analysis: what the data forced on us

Findings from the exploratory analysis (v1) and the depot-shift work (v7, v22–v26). The cell below recomputes the main ones.

1. **Rare, heavy-tailed target.** 397 failures out of 56,000 (0.71%). Repair cost: median 176, mean 492, maximum 8,000; the top 10% of
   failures carry 53% of all repair cost.
2. **A shifted histogram.** Sensors s054–s063 are a 10-bin histogram that is shifted by one bin in test: test bin $k+1$ matches train
   bin $k$. Every model realigns it. s047 (all missing in test) and s146 (constant) are dropped.
3. **A new depot.** After that fix a classifier still separates train from test (adversarial AUC 0.9999). s120 runs about +16M higher,
   s122 about ×2.7 at every quantile, and wear (s126, s128) is higher. Importance weighting is impossible (effective sample size
   0.3% of rows).
4. **A much higher failure rate at test.** The classifier predicts about 3× more failures on test; the label-shift EM estimate and the
   public scores put the test failure odds at roughly 3–6× training. This is why the flag rule had to be recalibrated (v7–v10).
5. **Censoring.** 22% of healthy vehicles were followed for less than 365 days. `followup_days` is partly determined by the outcome, so
   it is never a feature.
6. **Missing values predict failure** (most-missing quartile fails at 2.06%), so NaNs are kept and handled by XGBoost.

In [3]:
raw_tr, raw_te, _ = load(raw=True)
ks_shift = [ks_2samp(raw_tr[f"s{54 + k:03d}"].dropna(), raw_te[f"s{55 + k:03d}"].dropna()).statistic for k in range(9)]
ks_same = [ks_2samp(raw_tr[f"s{54 + k:03d}"].dropna(), raw_te[f"s{54 + k:03d}"].dropna()).statistic for k in range(9)]
fc = tr.repair_cost[tr.failure == 1]
print(f"failures {int(tr.failure.sum())} of {len(tr)} ({tr.failure.mean():.2%}); repair cost median {fc.median():.0f}, mean {fc.mean():.0f}, max {fc.max():.0f}")
print(f"histogram s054-s063, KS(train bin k, test bin k):   {np.round(ks_same, 3)}")
print(f"histogram s054-s063, KS(train bin k, test bin k+1): {np.round(ks_shift, 3)}")
q = [0.1, 0.5, 0.9]
for s in ("s120", "s122", "s126"):
    print(f"{s}: train quantiles {np.round(np.nanquantile(tr[s], q) / 1e6, 2)}M, test {np.round(np.nanquantile(te[s], q) / 1e6, 2)}M")
cens = (tr.followup_days < 365) & (tr.failure == 0)
print(f"healthy vehicles followed < 365 days: {cens.sum()} ({cens.sum() / (tr.failure == 0).sum():.0%})")

failures 397 of 56000 (0.71%); repair cost median 176, mean 492, max 8000
histogram s054-s063, KS(train bin k, test bin k):   [0.982 0.719 0.56  0.277 0.124 0.459 0.414 0.607 0.804]
histogram s054-s063, KS(train bin k, test bin k+1): [0.026 0.041 0.03  0.067 0.032 0.026 0.02  0.015 0.039]
s120: train quantiles [ 0.11 10.18 34.22]M, test [16.14 26.68 62.54]M
s122: train quantiles [0.   0.03 0.14]M, test [0.   0.08 0.48]M
s126: train quantiles [0.02 1.72 5.75]M, test [0.02 1.79 7.78]M
healthy vehicles followed < 365 days: 12088 (22%)


## 3. From the baseline to the final models

Our uploads in order. Each changed one thing, so the public score shows what that change was worth (the public difference between two
close files has noise of a few hundred points, so small differences are not evidence on their own). Rows marked ★ are the selected models.

| # | Version | What changed | Public |
|---|---|---|---|
| 1 | v2 | **Baseline:** XGBoost failure classifier, flag if p > 0.5, budget = mean repair cost | 74,933 |
| 2 | ★ **v3 (Model 1)** | Flag rule and budget derived from the cost function (Section 1) | **27,760** |
| 3 | v3b | v3 without the four drifting sensors; reverted | 29,530 |
| 4 | v4 | Platt calibration of the probabilities | 23,876 |
| 5–6 | v5 | Per-vehicle repair-cost model (XGBoost on log cost) for budgets and flags | 26,061 / 22,969 |
| 7–10 | v7, v9 | Depot shift: failure odds raised by a factor k (EM estimate); bigger cost model with histogram-shape features | 19,601 → 17,014 |
| 11 | v10 | k averaged over its posterior given the public scores | 17,349 |
| 12 | v12 | Repair-cost model without the four drifted sensors | 16,840 |
| 13–17 | v13–v17 | Seed averaging, quantile-mapping s120/s122, budget and flag-level probes | 16,826 – 17,975 |
| 18–20 | v18, v22, v24 | Rankings that correct for censoring or drop s120/s122: all worse publicly | 17,173 – 17,624 |
| 21 | v29 | **Harness:** Bayes decision under a posterior over test-depot hypotheses fitted to the public scores | 17,196 |
| 22–23 | v32, v33 | Histogram-kernel and **Gaussian-process repair-cost** budgets on v12-style flags (Daksh) | 16,992 / 16,588 |
| 24 | ★ **v34 (Model 2)** | v33 with budgets from each vehicle's GP uncertainty and a context-corrected failure probability (Daksh) | 16,588 |
| 25 | v35 | Harness flags with Gaussian-process budgets | 16,878 |
| 26 | v37 | Gaussian-process costs with wear and sensor groups, used for flags *and* budgets | 16,410 |
| 27 | v42 | Re-selected sensor group, cost spread by predicted-cost band | 16,487 |
| 28 | ★ **v51 (Model 3)** | Gaussian process with fewer inputs (ablation) | **16,388** |
| 29 | v82 (final upload) | Model 3's costs; decision hedged over all ranking families, incl. survival and CatBoost rankings | 16,440 |

**The steps that mattered.**

1. **Deriving the decision from the cost (v2 → v3, −63%).** A missed failure costs about 50 inspections, so the right threshold is near
   p = 2.5%, not 0.5. This was the largest single lever, and it is Model 1.
2. **Calibration and a per-vehicle cost model (v3 → v5, −17%).** Budgets became vehicle-specific instead of one global quantile.
3. **The depot shift (v5 → v12, −27%).** The test depot fails 3–6× more often. Raising the failure odds by k (EM label-shift estimate,
   then a posterior over k from the public scores) fixed the flag count. Removing the drifted sensors from the repair-cost model made
   the cost predictions robust to the new depot.
4. **A Gaussian-process repair-cost model (v12 → v33/v34, −1.5%).** A GP on the shapes of the sensor histograms predicts repair cost
   better than trees (out-of-fold R² 0.81 vs 0.80, budget loss −22%). Model 2 refines its budgets.
5. **The GP inside a Bayes decision fitted to the public scores (v29 → v51, −5% from v29).** The harness weighs hypotheses about the
   test depot by our earlier public scores; v37, v42 and v51 improve the GP inputs and spread. Model 3 is the result.

**What did not help** (each tried and measured; details in `../reports/findings.md`): dropping the drifted sensors from the classifier
(v3b, v22), censoring corrections (v18, v24), seed averaging (v13), quantile-mapping the drifted sensors (v14, v15), other model families
(CatBoost, LightGBM, histogram-gradient boosting, kernel methods), cost-weighted training, survival (AFT) rankings, and many repair-cost
variants (Section 9).

## 4. Why these models and these features

Every choice below was made by measuring it: on the exact competition cost in cross-validation, on the locked hold-out where noted,
and on the public board when two files differed by one change.

### 4.1 The decision layer: derived, not learned

The cost is asymmetric (a missed failure costs about 50 inspections, a unit of shortfall 10× a unit of unused budget), so we do not
tune a classification threshold or a budget multiplier. We compute, per vehicle, the flag and budget that minimise **expected cost**
(Section 1). A model therefore only has to supply a calibrated failure probability and a repair-cost distribution, and each can be judged
on its own. This choice alone took the public score from 74,933 to 27,760 (Model 1).

### 4.2 The failure model: XGBoost on the raw sensors

**Why gradient-boosted trees (XGBoost).**
- 168 numeric sensors with heavy tails, counters on very different scales, and histogram bins. Trees are unaffected by monotone
  transforms, so no scaling or log choices are needed, and they learn interactions between bins and counters without hand-made features.
- **Missing values carry signal** (vehicles in the most-missing quartile fail at 2.06%). XGBoost learns a default direction for missing
  values at each split, so NaNs are kept rather than imputed.
- Only 397 failures (0.71%), so the model is kept small: 300 trees, depth 4, learning rate 0.05, row and column subsampling 0.8. A sweep
  of 11 regularisation variants (depth 2/3/6, slower learning, larger leaves, L1/L2, `max_delta_step`, `scale_pos_weight`, gamma) found
  none better under seven validation designs (`src/v79_regular.py`).
- No class reweighting (`scale_pos_weight`): it distorts the probabilities that the decision needs, and it was worse in validation.

**Alternatives tried and rejected.** Logistic regression and kernel models (+0.73 cost per vehicle, `src/v37_riskkernel.py`); CatBoost,
LightGBM and sklearn's histogram gradient boosting alone (worse in most validation designs, HistGB up to +55%); seed averaging (v13,
worse publicly). Survival (AFT) models that use `followup_days` as a training target were better offline (−2% to −12%), but they rank the
test depot very differently and the public scores cannot tell which is right, so they enter only as one hypothesis among many (v82).

**Probability calibration (Platt).** The flag decision depends on $p$ itself around 2.5%, and the raw model is under-confident exactly
there: for raw $p$ between 0.003 and 0.028 the observed failure rate is 2.5% against 1.0% predicted. Platt scaling (a logistic fit on the
logit, two parameters) fixes this and cut CV cost from 1.242 to 1.107 per vehicle (4/5 folds). Isotonic regression did slightly better in
CV but is the riskier choice with 318 development failures and was the worse of the two on the hold-out, so Platt was kept.
Public: 27,760 → 23,876 (v4).

**The depot shift (prior shift).** The test depot fails 3–6× more often. We assume the sensors-given-outcome relationship carries over
and only the failure rate changes (label shift), and raise the failure odds by a factor $k$: estimated by EM (Saerens et al.), validated on
simulated shifts inside the development set (it recovers $k$ = 1, 2, 4, 6 as 0.99, 1.93, 3.89, 5.91), then averaged over a posterior from the public
scores. Importance weighting, the usual alternative for covariate shift, was impossible: train and test overlap so little that the
effective sample size was 0.3% of rows.

### 4.3 Failure-model features

- **All sensors, after three data fixes:** the s054–s063 histogram realigned (without it, test bin $k$ would be read as train bin $k$:
  KS up to 0.98 instead of ≤ 0.07); s047 dropped (all missing in test); s146 dropped (constant).
- **The drifted sensors (s120, s122, s126, s128) are kept** in the classifier. Dropping them looked neutral in CV but was worse publicly
  (v3b: +1,770; v22: +333), and quantile-mapping them to the training distribution was also worse (v14, v15). Part of the drift is genuine
  risk (wear is higher at the new depot). How much weight they deserve is left to the harness, whose hypotheses include rankings with and
  without them and explicit wear tilts.
- **`followup_days` is never a feature:** it is missing from test and partly determined by the outcome (records stop near failures).
- **Engineered features did not help the classifier:** missing-value counts (v6), histogram shares, means, entropies and tail shares
  (v8), per-usage rates, monotone constraints and linear terms (v60), and a duplicate-row flag (v72) were all within noise or worse.

### 4.4 The repair-cost model: a Gaussian process on histogram shapes

**Why a Gaussian process.**
- Repair cost is known only for the 397 failures. A GP is a smooth non-parametric regressor with a handful of hyperparameters (one
  amplitude per input group and a noise level), fitted by maximum marginal likelihood, so it overfits less on 397 points than boosted
  trees, and it gives a predictive distribution, not just a mean.
- **The target is log cost** (costs run from 5 to 8,000, heavily skewed). Budgets come from quantiles of out-of-fold residuals, so there is
  no back-transform bias.
- **The inputs are histogram shapes.** Repair cost tracks the *shape* of the sensor histograms (s080–s085 correlate about 0.55 with log
  cost). Each histogram becomes square-root bin shares (a Hellinger-type distance compares shapes, not raw counts, which scale with
  usage) plus cumulative shares (where in the histogram the mass sits). One RBF kernel per group, added together, lets each histogram
  contribute its own smooth effect, and the fitted amplitudes act as automatic relevance weights.
- Evidence: out-of-fold R² 0.814 against 0.798 for the tree model, and budget loss −22% (v35); ridge, SVR, kNN and gradient boosting
  were 56–113% worse on budget loss, and every blend with the GP was worse (`src/v84_costens.py`).

**Cost-model inputs.**
- **The drifted sensors are excluded** from the cost model: costs predicted from them extrapolate at the new depot (v12: public −509).
- **Wear group** (log s126, s149, s123, Model 3): more-worn vehicles cost more to repair, and the test depot is more worn.
- **Cost sensors** (s116, s142, s105, s104, s078, Model 3): chosen by a greedy search on budget loss, and kept only if stable between
  depots (KS ≤ 0.032), so they do not shift at the new depot.
- **Removed by ablation** (Model 3): the usage group and the s023 and s129 histograms made budgets worse when removed one at a time;
  s080, s054, s037 and s154 are essential (removing them costs +7% to +38%).
- **Spread of the cost distribution:** Model 2 scales each vehicle's residuals by the GP's own predictive uncertainty; Model 3 takes
  residual quantiles from failures with similar predicted cost (three bands), which was more stable in our checks (the per-vehicle
  uncertainty barely tracked the actual errors, correlation 0.02–0.05).

### 4.5 Why these three models

- **Model 1 (v3)** isolates the most important idea: the decision derived from the cost, on a plain classifier.
- **Model 2 (v34)** is the strongest version of the single-classifier pipeline: calibration, depot shift, and Gaussian-process budgets
  with per-vehicle uncertainty, with the flags fixed from v33.
- **Model 3 (v51)** adds a decision fitted to the evidence about the test depot and the refined cost model, and has our best public score.

## 5. Model 1: v3, the decision derived from the cost (public 27,760.14)

**Idea.** The baseline v2 used a reasonable classifier with the wrong decision: flag if p > 0.5 and a fixed budget. v3 keeps the same
classifier and replaces the decision with the rule of Section 1. Here $F$ is **global**: the empirical cost distribution of the
training failures, the same for every vehicle.

- **Classifier:** XGBoost (300 trees, depth 4, learning rate 0.05) on all sensors after the v1 data fixes.
- **Validation:** 5 fixed stratified folds of the 80% development split; the 20% hold-out stays locked. Cost per vehicle with the exact
  competition cost. In each fold the budget distribution uses only the training folds' failures.

The cell reports the cross-validated cost of the v2 rule and of v3 on the same folds, then writes the test file.

In [4]:
oof = fit_oof(dev[S])
p_te = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
naive = fold_cost((oof > 0.5).astype(int), np.full(len(y), c[c > 0].mean()))
fl, bu = np.zeros(len(y), int), np.zeros(len(y))
for a, b in F:
    fl[b], bu[b] = decide_global(oof[b], c[a][c[a] > 0])
v3cv = fold_cost(fl, bu)
print(f"v2 rule (p > 0.5): {naive.mean():.3f} ± {naive.std():.3f} per vehicle")
print(f"v3 rule          : {v3cv.mean():.3f} ± {v3cv.std():.3f} per vehicle; paired change per fold {np.round(v3cv - naive, 3)}; flagged {fl.mean():.2%}")
f3, b3 = decide_global(p_te, c_all[c_all > 0])
write_submission(te.id, f3, b3, "v3")
SUMMARY = {"Model 1 (v3)": dict(flags=int(f3.sum()), mean_budget=b3[f3 == 1].mean())}

v2 rule (p > 0.5): 2.501 ± 0.833 per vehicle
v3 rule          : 1.242 ± 0.387 per vehicle; paired change per fold [-0.704 -1.778 -0.839 -1.041 -1.932]; flagged 2.08%
wrote outputs/submission_v3.csv: flagged 5.37% of test vehicles


**Intermediate results.**

| | CV cost / vehicle | Public score |
|---|---|---|
| v2 (p > 0.5) | 2.50 | 74,933 |
| **v3** | **1.24** | **27,760** (−63%) |

The cost halves in every fold, and the public score falls by 63%. About 5.4% of test vehicles are flagged (v2: 1.6%), as expected from
a threshold near 2.5%.

**What v3 gets wrong, which motivated the next versions.** The decision depends on $p$ itself near 2.5%, so it needs **calibrated**
probabilities (v4). $F$ is the same for every vehicle, although repair cost is predictable from the sensors (v5). And it assumes the
training depot's failure rate, while the new depot fails 3–6× more often (v7–v10).

## 6. Model 2: v34, Gaussian-process budgets with per-vehicle uncertainty (Daksh)

**Starting point: v33.** v33 keeps the v12-era decisions for *which* vehicles to flag: Platt-calibrated XGBoost probabilities $q$, the
depot-shift correction $p = 3q/(1 - q + 3q)$ (failure odds ×3), and the tree repair-cost model. It replaces the **budgets** with a
Gaussian-process repair-cost model, multiplied by 1.25 (16,588 public, −252 against v12-style files).

**The Gaussian process (v33).** Fitted on the 397 training failures, on log repair cost. Each of the six sensor histograms becomes a shape
vector (square-root bin shares and cumulative shares); the kernel is a sum of one Gaussian (RBF) kernel per histogram plus a usage total,
with amplitudes and noise fitted by maximum marginal likelihood and lengthscales set by the median pairwise distance. Vehicles with too
few observed histograms are priced by the tree model.

**v34 changes only the budgets of the 1,780 flagged vehicles**, in two ways:

1. **Per-vehicle uncertainty.** The GP also returns each vehicle's predictive standard deviation $\sigma_i$. The budget becomes
   $\exp(\mu_i + \sigma_i\,z_\ell)$, where $z_\ell$ is the newsvendor-level quantile of the *standardised* out-of-fold residuals, so a vehicle
   unlike the training failures gets a wider budget.
2. **A context correction to the failure probability, for budgets only.** A small penalised logistic regression with logit$(q)$ as an
   offset, on wear (s126), usage (s080–s089), their difference, s123, s149 and the missing-value fraction, shifts each vehicle's
   probability by at most ±1.5 on the logit scale. The budget level $1 - 1/(11p)$ depends on $p$, so this changes budgets but not flags.

**Intermediate results** (Daksh's validation, `reports/v34_implementation_plan.md`; 44,800 development vehicles, 5 outer folds × 3
repeats, simulated depot with 3× failure weight and the test depot's wear mix):

| check | result against v33 |
|---|---|
| per-vehicle uncertainty alone | −1.54% simulated depot cost |
| budget-only policy (v34), ordinary cost | 0.8288 → 0.8246 per vehicle |
| budget-only policy (v34), simulated depot cost | 1.9807 → 1.9421 (−1.96%, −1.65%, −2.24% over 3 repeats) |
| extrapolation: train without the top usage decile | +0.42% (worse) |
| extrapolation: train without the top wear decile | −0.14% |
| test-like wear × usage mix | −0.76% |
| rejected along the way | learned lengthscales +1.39%, histogram interactions +1.15%, log-ratio kernels +1.71%, no ×1.25 budget multiplier +1.65% |

The full policy (changing flags too) cost more on the hold-out, so v34 keeps v33's flags exactly.

**Artifacts:** `../reports/v33_baseline.npz` (v12 probabilities $q$, tree log-cost and its residuals; `src/v33_build.py`),
`../reports/p_oof_v2.npy` (out-of-fold classifier probabilities for the correction), and `../submissions/submission_v33_gp_budget.csv`,
whose budgets v34 keeps for unflagged vehicles (the scorer ignores those). Builder: `src/v34_build.py`.

In [5]:
"""Small additive Gaussian process for ordered sensor histograms, research only."""
import numpy as np
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

START = (23,37,54,80,129,154)

def extract(data):
    groups=[]
    for start in START:
        width=9 if start==54 else 10
        x=data[[f"s{i:03d}" for i in range(start,start+width)]].to_numpy(float)
        total=x.sum(axis=1)
        ok=np.isfinite(x).all(1)&(x>=0).all(1)&(total>0)
        frac=x/np.where(ok,total,np.nan)[:,None]
        # Shape and within-histogram location both vary continuously.
        groups.append(np.column_stack([np.sqrt(frac),frac.cumsum(1)[:,:-1]]))
        if start==80: exposure=np.log1p(np.where(ok,total,np.nan))[:,None]
    return groups+[exposure]

def select(data,rows):
    return [x[rows] for x in data]

class GaussianSeverity:
    def blocks(self, features):
        result=[]
        for x,base,scale in zip(features,self.training,self.scale):
            distance=cdist(x,base,'sqeuclidean')
            result.append(np.nan_to_num(np.exp(-distance/(2*scale)),nan=0.))
        return np.asarray(result)

    def fit(self,features,y):
        self.training=features
        self.scale=[]
        ind=np.triu_indices(len(y),1)
        for x in features:
            ds=cdist(x,x,'sqeuclidean')[ind]
            ds=ds[np.isfinite(ds)]
            md=np.median(ds) if len(ds) else 1.
            self.scale.append(max(md,1.e-6))
        blocks=self.blocks(features)
        self.center=np.mean(y)
        z=y-self.center
        unit=np.eye(len(y))
        def value(logtheta,final=False):
            theta=np.exp(logtheta)
            cov=np.einsum('i,ijk->jk',theta[:-1],blocks)+theta[-1]*unit
            factor=cho_factor(cov,lower=True,check_finite=False)
            alpha=cho_solve(factor,z,check_finite=False)
            if final: return alpha
            loss=.5*z@alpha+np.log(np.diag(factor[0])).sum()+.5*(logtheta@logtheta)/4
            inv=cho_solve(factor,unit,check_finite=False)-np.outer(alpha,alpha)
            grad=.5*theta*np.r_[np.einsum('ij,kij->k',inv,blocks),np.trace(inv)]+logtheta/4
            return loss,grad
        res=minimize(value,np.zeros(len(blocks)+1),jac=True,method='L-BFGS-B',
                     bounds=[(-5.,5.)]*len(blocks)+[(-4.,3.)],options={'maxiter':35})
        self.amplitude=np.exp(res.x[:-1])
        self.alpha=value(res.x,True)
        self.bounds=(min(y),max(y))
        return self

    def predict(self,features):
        out=np.full(len(features[0]),np.nan)
        for start in range(0,len(out),800):
            rows=slice(start,start+800)
            vals=select(features,rows)
            valid=sum(np.isfinite(x).all(1).astype(int) for x in vals[:-1])>=3
            blocks=self.blocks(vals)
            pred=np.einsum('i,ijk->jk',self.amplitude,blocks)@self.alpha+self.center
            out[rows]=np.where(valid,np.clip(pred,*self.bounds),np.nan)
        return out

In [6]:
from scipy.special import expit, logit as slogit

class GaussianSeveritySD:   # Daksh's v34 class from src/v34_hist_gp.py, verbatim apart from its name
    def blocks(self, features):
        result=[]
        for x,base,scale in zip(features,self.training,self.scale):
            distance=cdist(x,base,'sqeuclidean')
            result.append(np.nan_to_num(np.exp(-distance/(2*scale)),nan=0.))
        return np.asarray(result)

    def fit(self,features,y):
        self.training=features
        self.scale=[]
        ind=np.triu_indices(len(y),1)
        for x in features:
            ds=cdist(x,x,'sqeuclidean')[ind]
            ds=ds[np.isfinite(ds)]
            md=np.median(ds) if len(ds) else 1.
            self.scale.append(max(md,1.e-6))
        blocks=self.blocks(features)
        self.center=np.mean(y)
        z=y-self.center
        unit=np.eye(len(y))
        def value(logtheta,final=False):
            theta=np.exp(logtheta)
            cov=np.einsum('i,ijk->jk',theta[:-1],blocks)+theta[-1]*unit
            factor=cho_factor(cov,lower=True,check_finite=False)
            alpha=cho_solve(factor,z,check_finite=False)
            if final:
                self.factor = factor
                self.noise = theta[-1]
                return alpha
            loss=.5*z@alpha+np.log(np.diag(factor[0])).sum()+.5*(logtheta@logtheta)/4
            inv=cho_solve(factor,unit,check_finite=False)-np.outer(alpha,alpha)
            grad=.5*theta*np.r_[np.einsum('ij,kij->k',inv,blocks),np.trace(inv)]+logtheta/4
            return loss,grad
        res=minimize(value,np.zeros(len(blocks)+1),jac=True,method='L-BFGS-B',
                     bounds=[(-5.,5.)]*len(blocks)+[(-4.,3.)],options={'maxiter':35})
        self.amplitude=np.exp(res.x[:-1])
        self.alpha=value(res.x,True)
        self.bounds=(min(y),max(y))
        return self

    def predict(self,features):
        out=np.full(len(features[0]),np.nan)
        spread=np.full(len(out),np.nan)
        for start in range(0,len(out),800):
            rows=slice(start,start+800)
            vals=select(features,rows)
            valid=sum(np.isfinite(x).all(1).astype(int) for x in vals[:-1])>=3
            blocks=self.blocks(vals)
            covariance = np.einsum('i,ijk->jk',self.amplitude,blocks)
            pred=covariance@self.alpha+self.center
            out[rows]=np.where(valid,np.clip(pred,*self.bounds),np.nan)
            diagonal = sum(weight * np.isfinite(group).all(axis=1)
                           for weight, group in zip(self.amplitude, vals))
            solved = cho_solve(self.factor, covariance.T, check_finite=False).T
            variance = diagonal + self.noise - np.einsum("ij,ij->i", covariance, solved)
            spread[rows] = np.where(valid, np.sqrt(np.maximum(self.noise, variance)), np.nan)
        return out, spread

In [7]:
REP = Path("../reports")
def quantile_budget(prob, mean, residuals, spread=1.0):
    out = np.zeros(len(prob)); pos = prob > 1 / 11; lev = 1 - 1 / (11 * prob[pos])
    out[pos] = np.exp(mean[pos] + np.broadcast_to(spread, prob.shape)[pos] * np.quantile(residuals, lev)); return out

# repair costs: GP mean and sd for every test vehicle; standardised out-of-fold residuals (5 folds, seed 42, as v33)
fails = tr[tr.failure == 1]; Xf = extract(fails); lyf = np.log(fails.repair_cost.to_numpy())
gm, gs = np.full(len(lyf), np.nan), np.full(len(lyf), np.nan)
for a, b in KFold(5, shuffle=True, random_state=42).split(lyf):
    gm[b], gs[b] = GaussianSeveritySD().fit(select(Xf, a), lyf[a]).predict(select(Xf, b))
okf = np.isfinite(gm); res34 = lyf[okf] - gm[okf]; std34 = res34 / gs[okf]
mean34, sigma34 = GaussianSeveritySD().fit(Xf, lyf).predict(extract(te)); sup34 = np.isfinite(mean34)
print(f"GP: {okf.sum()} out-of-fold failures, R2 {1 - (res34 ** 2).sum() / ((lyf[okf] - lyf[okf].mean()) ** 2).sum():.4f}; "
      f"corr(|residual|, sd) {np.corrcoef(np.abs(res34), gs[okf])[0, 1]:.3f}; test vehicles priced by the GP {sup34.sum()}")

# flags exactly as v33: v12 probabilities with the prior shift k = 3 and the tree cost model
base = np.load(REP / "v33_baseline.npz"); assert np.array_equal(base["ids"], te.id)
q33, tree33, tres33 = base["q"], base["tree"], base["residual"]
p33 = 3 * q33 / (1 - q33 + 3 * q33)
fl34, _ = decide_pv(p33, tree33, tres33); insp = fl34 == 1
print(f"{insp.sum()} flags (the v33 inspection set)")

# change 1: context correction to the failure odds (wear, usage, missingness), used for budgets only
MISS = [f"s{i:03d}" for i in range(170) if i not in {47, 63, 120, 122, 146}]; USAGE = [f"s{i:03d}" for i in range(80, 90)]
def context(d):
    wear = np.log1p(d.s126.to_numpy(float)); use = np.log1p(d[USAGE].sum(axis=1, min_count=10).to_numpy(float))
    return np.column_stack([wear, use, wear - use, np.log1p(d.s123), np.log1p(d.s149), d.reindex(columns=MISS).isna().mean(axis=1)])
raw_te2 = raw_te.copy(); raw_te2[HISTB[:-1]] = raw_te2[HISTB[1:]].to_numpy(); raw_te2[HISTB[-1]] = np.nan
dev_raw = raw_tr.iloc[dev_idx].reset_index(drop=True)
p_oof = np.load(REP / "p_oof_v2.npy"); qdev = Platt().fit(p_oof, y).predict(p_oof)
Fd = context(dev_raw); lo, med, hi = np.nanquantile(Fd, [0.01, 0.5, 0.99], axis=0)
fill = lambda v: np.clip(np.where(np.isfinite(v), v, med), lo, hi)
Td = fill(Fd); cen, scl = Td.mean(0), np.maximum(Td.std(0), 0.001)
design = lambda v, t: np.column_stack([np.ones(len(v)), (t - cen) / scl, np.isnan(v[:, :2])])
Xd = design(Fd, Td); off = slogit(np.clip(qdev, 1e-6, 1 - 1e-6))
def nll(cf):
    s = off + Xd @ cf; return np.sum(np.logaddexp(0, s) - y * s) + 8 * cf @ cf, Xd.T @ (expit(s) - y) + 16 * cf
coef = minimize(nll, np.zeros(Xd.shape[1]), jac=True, method="L-BFGS-B").x
Ft = context(raw_te2); eta = np.clip(design(Ft, fill(Ft)) @ coef, -1.5, 1.5)
p_budget = expit(slogit(np.clip(p33, 1e-9, 1 - 1e-9)) + eta)
print(f"context correction on flagged vehicles: logit shift mean {eta[insp].mean():+.3f}, range {eta[insp].min():+.2f} to {eta[insp].max():+.2f}")

# change 2: each vehicle's GP sd scales the standardised residual quantiles; budgets x1.25 as in v33
bu34 = np.zeros(len(te)); act = insp & sup34; fb = insp & ~sup34
bu34[act] = (1.25 * quantile_budget(p_budget[act], mean34[act], std34, sigma34[act])).round(2)
bu34[fb] = (1.25 * quantile_budget(p_budget[fb], tree33[fb], tres33)).round(2)
v33file = pd.read_csv(Path("../submissions/submission_v33_gp_budget.csv")).set_index("id").loc[te.id]
bu34[~insp] = v33file.budget.to_numpy()[~insp]          # unflagged rows keep v33's values (the scorer ignores budgets of unflagged vehicles)
print(f"mean budget on flagged vehicles {bu34[insp].mean():.0f} (v33: {v33file.budget.to_numpy()[insp].mean():.0f})")
write_submission(te.id, fl34, bu34, "v34_budget")
SUMMARY["Model 2 (v34)"] = dict(flags=int(insp.sum()), mean_budget=bu34[insp].mean())

GP: 383 out-of-fold failures, R2 0.8137; corr(|residual|, sd) 0.025; test vehicles priced by the GP 19386
1780 flags (the v33 inspection set)
context correction on flagged vehicles: logit shift mean +0.013, range -0.52 to +0.13
mean budget on flagged vehicles 571 (v33: 571)
wrote outputs/submission_v34_budget.csv: flagged 8.90% of test vehicles


## 7. Model 3: v51, a Bayes decision fitted to the public scores, with a reduced-input Gaussian process (public 16,388.51, best)

Model 3 changes both halves of the decision.

**Failure probabilities: the harness.** Which ranking holds at the test depot, and how much more often it fails, cannot be learned from
training data. The harness considers a family of hypotheses about the test depot:

$$p_i = \sigma\!\left(a + b\,L_i + c\,z_i\right),$$

where $L_i$ blends the logits of ten classifier rankings built earlier (v12, no-s122, soft labels, censoring ensembles,
instrument-corrected, quantile-mapped, ...), $z_i$ is a wear score (s120, s122, s126 or s149) with tilt $c$, $b$ is a confidence slope,
$a$ the overall level, and a repair-cost level $\alpha$ scales costs. Each hypothesis is weighted by the **exact likelihood of our earlier
public scores**: for a hypothesis we know the mean and covariance of every uploaded file's score on a random 30% of test, so the observed
scores give a Gaussian likelihood. The decision is the **Bayes decision**: for each vehicle, the flag and newsvendor budget that minimise
the posterior-expected cost. Budgets are capped at 8,000.

**Repair costs: three refinements of the v33 Gaussian process**, each checked with honest cross-fitted budget loss (budget quantiles
estimated on other folds' failures, weighted over the newsvendor levels the flagged test vehicles need):

| step | change | evidence | public |
|---|---|---|---|
| v37 | add a wear group (s126, s149, s123) and six cost sensors; use GP costs for flags too | R² 0.814 → 0.832, budget loss −4.6% | 16,410 |
| v42 | sensors re-selected (s116, s142, s105, s104, s078); residual spread by predicted-cost third | −7.9%, −6.6%, −6.2% on 3 fold seeds | 16,487 |
| **v51** | **drop the usage group and the s023 and s129 histograms; average fits with and without s123** | **−7.4% (10/10 fresh seeds); end-to-end CV −2.15% (18/25 folds)** | **16,388** |

The v51 inputs came from an ablation (`src/v47_ablation.py`, 26 variants, 5 seeds): removing histogram s080 costs +38% budget loss, s054
+20%, s037 +10%, s154 +7%, the sensor group +6%; Matérn or rational-quadratic kernels, raw bin levels, other cost transforms, other
sensors, principal components and a missing-value count were no better; removing the usage group and the s023 and s129 histograms helped.
The groups were chosen on the same failures, so the real gain is probably smaller than measured. End-to-end, GP costs also beat tree
costs in the flag decision: −3.3% cost per vehicle (11/15 folds, `src/v45_flagmu.py`).

**Artifacts** (`../reports/`): `v18_qm.npy` (v12 test probabilities and tree log-cost, `src/v18_build.py`), `v18_resid.npy`, the ranking
probabilities `v19_q.npy`, `v21_H.npy`, `v22_q.npy`, `v24_q.npy`, `v24_pu_x2_q.npy`, `v18_ens10_q.npy`, `v18_ens_q.npy`, `p_oof_v2.npy`,
and the posterior `v51_post.csv` (26 public scores, `src/v51_harness.py`).

In [8]:
REP, SUB = Path("../reports"), Path("../submissions")
qv12, m = np.load(REP / "v18_qm.npy"); resid = np.load(REP / "v18_resid.npy")
sig = lambda v: 1 / (1 + np.exp(-v)); logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
def zs(x): x = np.log1p(np.nan_to_num(np.maximum(x, 0), nan=np.nanmedian(x))); return (x - x.mean()) / x.std()
v19 = np.load(REP / "v19_q.npy"); H21 = np.load(REP / "v21_H.npy")
te_map = te.copy()
for col in ("s120", "s122"):                                                        # quantile-map the two test sensors onto train
    r = te[col].rank(pct=True); te_map[col] = np.nanquantile(tr[col], r.fillna(0.5).clip(0, 1)); te_map.loc[te[col].isna(), col] = np.nan
mdl = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure)
q_map = Platt().fit(np.load(REP / "p_oof_v2.npy"), y).predict(mdl.predict_proba(te_map[S])[:, 1])
RANK = {"v12": qv12, "nos122": np.load(REP / "v22_q.npy"), "pu": np.load(REP / "v24_q.npy"), "pu_x2": np.load(REP / "v24_pu_x2_q.npy"),
        "ens10": np.load(REP / "v18_ens10_q.npy"), "ens4": np.load(REP / "v18_ens_q.npy"), "fix120": v19[0], "fix": v19[1],
        "nodrift": H21[2], "map": q_map}                                            # ten ranking hypotheses
RANK = {k: logit(np.ravel(v)[-len(te):]) for k, v in RANK.items()}
Z = {"none": np.zeros(len(te)), **{s: zs(te[s].to_numpy()) for s in ("s120", "s122", "s126", "s149")}}   # wear scores

def prob(r):
    """failure probabilities of one 'truth': blend of rankings, wear tilts, confidence slope b, level a"""
    g = lambda k, d: r[k] if k in r and isinstance(r[k], (int, float, str)) and r[k] == r[k] else d
    w2, R2, c2, z2 = g("w2", 0.0), g("R2", "v12"), g("c2", 0.0), g("z2", "none")
    L = (1 - r["w"] - w2) * RANK["v12"] + r["w"] * RANK[r["R"]] + w2 * RANK[R2]
    return r["fl0"] + (1 - r["fl0"]) * sig(r["a"] + r["b"] * L + r["c"] * Z[r["z"]] + c2 * Z[z2])
ALPHAS = (1.0, 1.15, 1.3, 1.45); Kd = 99; grid = (np.arange(Kd) + .5) / Kd          # repair-cost level multipliers, 99 cost draws

In [9]:
def posterior_means(T):
    """posterior-weighted failure probability and weight, per cost level alpha"""
    T = T.copy(); T.wt /= T.wt.sum()
    PB = {al: sum((r.wt * prob(r._asdict()) for r in T[T.alpha == al].itertuples()), np.zeros(len(te))) for al in ALPHAS}
    return PB, {al: T[T.alpha == al].wt.sum() for al in ALPHAS}

def bayes_decide(PB, Wa, draws):
    """flag and budget that minimise posterior-expected cost; budgets capped at 8,000 (the largest repair cost seen)"""
    MX = {al: np.clip(draws * al, 5, 8000) for al in ALPHAS}
    wsum = sum(Wa.values()); p = sum(PB.values()) / wsum
    Mx = np.concatenate([MX[al] for al in ALPHAS if Wa[al] > 0], 1)
    wcol = np.concatenate([PB[al][:, None] / np.maximum(p[:, None] * wsum, 1e-12) * np.ones((1, Kd)) / Kd for al in ALPHAS if Wa[al] > 0], 1)
    o = np.argsort(Mx, 1); Ms = np.take_along_axis(Mx, o, 1); cw = np.cumsum(np.take_along_axis(wcol, o, 1), 1)
    Q = 1 - (1 - Q_STAR) / np.maximum(p, 1e-9)
    idx = np.clip((cw < np.clip(Q, 0, 1)[:, None]).sum(1), 0, Ms.shape[1] - 1)
    b = np.minimum(np.where(Q > 0, Ms[np.arange(len(p)), idx], 0.0), 8000)
    fail = ((UNDER * np.maximum(Mx - b[:, None], 0) + OVER * np.maximum(b[:, None] - Mx, 0)) * wcol).sum(1); mu = (Mx * wcol).sum(1)
    fl = (INSPECT + p * fail + (1 - p) * OVER * b < p * mu).astype(int); return fl, np.where(fl == 1, b, 0.0)

def load_posterior(path):
    P = pd.read_csv(REP / path)
    print(f"{len(P)} truths; weight by wear tilt:", P.groupby("z").wt.sum().round(3).to_dict())
    return P[P.wt > 1e-4]

In [10]:
class GPAll(GaussianSeverity):
    """predicts every row; which rows count is decided below, the same way for both fits"""
    def predict(self, features):
        return np.clip(np.einsum("i,ijk->jk", self.amplitude, self.blocks(features)) @ self.alpha + self.center, *self.bounds)

f = tr[tr.failure == 1].reset_index(drop=True); ly = np.log(f.repair_cost.to_numpy())
START6 = (23, 37, 54, 80, 129, 154); KEEP = (37, 54, 80, 154)
SENS = ["s116", "s142", "s105", "s104", "s078"]
def z(d, cols):                                   # log1p, standardised with the training failures' mean and sd
    R = np.log1p(np.maximum(f[cols].to_numpy(float), 0))
    return (np.log1p(np.maximum(d[cols].to_numpy(float), 0)) - np.nanmean(R, 0)) / (np.nanstd(R, 0) + 1e-9)
def feats(d, wear):
    H = extract(d)
    return [H[START6.index(a)] for a in KEEP] + [z(d, wear), z(d, SENS)]
# rows the Gaussian process may price: enough of the original blocks observed
valid = sum(np.isfinite(x).all(1).astype(int) for x in extract(f)[:6]) >= 3
sup = sum(np.isfinite(x).all(1).astype(int) for x in extract(te) + [z(te, ["s126", "s149", "s123"])]) >= 3
oofs, sevs = [], []
for wear in (["s126", "s149", "s123"], ["s126", "s149"]):
    X = feats(f, wear); o = np.full(len(ly), np.nan)
    for a, b in KFold(5, shuffle=True, random_state=42).split(ly):
        o[b] = GPAll().fit(select(X, a), ly[a]).predict(select(X, b))
    o[~valid] = np.nan; s = GPAll().fit(X, ly).predict(feats(te, wear)); s[~sup] = np.nan
    oofs.append(o); sevs.append(s)
oof, sev = (oofs[0] + oofs[1]) / 2, (sevs[0] + sevs[1]) / 2
ok = np.isfinite(oof); gres = ly[ok] - oof[ok]; sup = np.isfinite(sev)
print(f"GP out-of-fold R2 {1 - (gres ** 2).sum() / ((ly[ok] - ly[ok].mean()) ** 2).sum():.4f}; residual sd {gres.std():.3f}; test supported {sup.sum()}")

GP out-of-fold R2 0.8385; residual sd 0.658; test supported 19386


In [11]:
P = load_posterior("v51_post.csv")
mu_oof = oof[ok]; edges = np.quantile(mu_oof, [1 / 3, 2 / 3])
band_of_failure = np.searchsorted(edges, mu_oof); band = np.searchsorted(edges, np.where(sup, sev, 0.0))
QB = np.stack([np.quantile(gres[band_of_failure == j], grid) for j in range(3)])
draws = np.where(sup[:, None], np.exp(np.where(sup, sev, m)[:, None] + QB[band]), np.exp(m[:, None] + np.quantile(resid, grid)[None, :]))
fl, bu = bayes_decide(*posterior_means(P), draws)
print(f"{int(fl.sum())} flags, mean budget {bu[fl == 1].mean():.0f}")
write_submission(te.id, fl, bu, "v51_gp_reduced")
SUMMARY["Model 3 (v51)"] = dict(flags=int(fl.sum()), mean_budget=bu[fl == 1].mean())

3520 truths; weight by wear tilt: {'none': 0.022, 's120': 0.062, 's122': 0.717, 's126': 0.099, 's149': 0.1}


1433 flags, mean budget 651
wrote outputs/submission_v51_gp_reduced.csv: flagged 7.17% of test vehicles


## 8. Performance across the three models, and a check against the uploaded files

The table is computed from the runs above. The check confirms that each regenerated CSV is identical to the file uploaded to Kaggle
(same ids and flags, budgets equal to the cent).

In [12]:
PUBLIC = {"Model 1 (v3)": 27760.14, "Model 2 (v34)": 16587.74, "Model 3 (v51)": 16388.51}     # public scores from Kaggle
R = pd.DataFrame(SUMMARY).T; R["public"] = pd.Series(PUBLIC)
R["public vs Model 1"] = (R.public / R.public.iloc[0] - 1).map(lambda v: f"{v:+.1%}" if v == v else "n/a")
print(R.round({"mean_budget": 0, "public": 2}).to_string()); print()
for name in ("v3", "v34_budget", "v51_gp_reduced"):
    a, b = pd.read_csv(OUT / f"submission_{name}.csv"), pd.read_csv(Path("../submissions") / f"submission_{name}.csv")
    same = a.id.equals(b.id) and (a.flag == b.flag).all() and np.allclose(a.budget, b.budget, atol=0.015)
    print(f"{'MATCH ' if same else 'DIFFER'} {name}: flags differ {int((a.flag != b.flag).sum())}, max budget difference {np.abs(a.budget - b.budget).max():.2f}")

                flags  mean_budget    public public vs Model 1
Model 1 (v3)   1073.0        403.0  27760.14             +0.0%
Model 2 (v34)  1780.0        571.0  16587.74            -40.2%
Model 3 (v51)  1433.0        651.0  16388.51            -41.0%

MATCH  v3: flags differ 0, max budget difference 0.00
MATCH  v34_budget: flags differ 0, max budget difference 0.00
MATCH  v51_gp_reduced: flags differ 0, max budget difference 0.00


**Reading the gains.**

- **Model 1 → Model 2.** The decision rule (Model 1) is the foundation. Model 2 adds everything that followed it: calibrated
  probabilities, the depot-shift correction (odds ×3), a per-vehicle repair-cost model, and on top of that the Gaussian-process budgets
  with per-vehicle uncertainty. Together these cut the public score from 27,760 to 16,588, −40%.
- **Model 2 → Model 3.** Model 3 changes *which* vehicles are flagged (about 1,430 instead of 1,780, chosen by a posterior fitted to the
  public scores) and uses a better GP (R² 0.84 against 0.81, with banded spread and reduced inputs). Offline the three GP refinements
  (v37, v42, v51) cut budget loss by roughly 15–18% relative to v33's GP. Publicly it is our best file: 16,388, −200 (−1.2%) against
  Model 2.
- On the public board, differences below about 300 are inside the noise: only about 190 public vehicles fail, with heavy-tailed costs.

## 9. Error analysis of the best model, its limitations and capabilities, and the leaderboard

The cell below breaks Model 3's expected cost on the public 30% into its parts, under its own posterior, and compares it with two
oracles: knowing which vehicles fail, and knowing each vehicle's repair cost if it fails.

In [13]:
PB, Wa = posterior_means(P); wsum = sum(Wa.values()); flg = fl == 1
parts = dict.fromkeys(["inspections", "unused budget, flagged vehicles that do not fail", "shortfall, flagged failures",
                       "unused budget, flagged failures", "missed failures (not flagged)"], 0.0)
extra = dict.fromkeys(["fail_flagged", "fail_missed", "oracle_fail", "oracle_cost"], 0.0)
for al in ALPHAS:
    if Wa[al] == 0: continue
    w = Wa[al] / wsum; p = PB[al] / Wa[al]; C = np.clip(draws * al, 5, 8000); mu = C.mean(1)
    sh = (UNDER * np.maximum(C - bu[:, None], 0)).mean(1); ov = (OVER * np.maximum(bu[:, None] - C, 0)).mean(1)
    parts["inspections"] += w * INSPECT * flg.sum()
    parts["unused budget, flagged vehicles that do not fail"] += w * (OVER * bu * (1 - p))[flg].sum()
    parts["shortfall, flagged failures"] += w * (p * sh)[flg].sum()
    parts["unused budget, flagged failures"] += w * (p * ov)[flg].sum()
    parts["missed failures (not flagged)"] += w * (p * mu)[~flg].sum()
    extra["fail_flagged"] += w * p[flg].sum(); extra["fail_missed"] += w * p[~flg].sum()
    extra["oracle_fail"] += w * (p * (INSPECT + sh + ov)).sum()                                  # flag exactly the failures, same budgets
    best = np.minimum(p[:, None] * UNDER * C, (1 - p)[:, None] * OVER * C).mean(1)              # know c if it fails: budget 0 or c
    extra["oracle_cost"] += w * (INSPECT * flg.sum() + (p * mu)[~flg].sum() + best[flg].sum())
tot = sum(parts.values())
print("Model 3, expected cost on the public 30% (actual public score 16,388.51):")
for k, v in parts.items(): print(f"  {k:50s} {0.3 * v:7.0f}  ({v / tot:5.1%})")
print(f"  {'total':50s} {0.3 * tot:7.0f}")
print(f"\nexpected failures on the public 30%: {0.3 * (extra['fail_flagged'] + extra['fail_missed']):.0f}, of which flagged "
      f"{extra['fail_flagged'] / (extra['fail_flagged'] + extra['fail_missed']):.0%}; flagged vehicles {0.3 * flg.sum():.0f}")
print(f"oracle, knowing which vehicles fail: {0.3 * extra['oracle_fail']:.0f};  oracle, knowing each repair cost: {0.3 * extra['oracle_cost']:.0f}")

Model 3, expected cost on the public 30% (actual public score 16,388.51):
  inspections                                           4299  (31.5%)
  unused budget, flagged vehicles that do not fail      2102  (15.4%)
  shortfall, flagged failures                           3513  (25.8%)
  unused budget, flagged failures                       1645  (12.1%)
  missed failures (not flagged)                         2074  (15.2%)
  total                                                13633

expected failures on the public 30%: 186, of which flagged 90%; flagged vehicles 430
oracle, knowing which vehicles fail: 7438;  oracle, knowing each repair cost: 9581


**What the breakdown says.** About a third of the expected cost is inspections, about half is budget error (shortfall plus unused
budget), and about a sixth is missed failures. The model catches about 90% of expected failures. The two oracles bound what is left:
knowing which vehicles fail would save roughly 6,000 on the public 30%; knowing repair costs exactly, about 4,000.

**Capabilities of the best model.**
- Decisions follow from the cost function: a 2.5% flag threshold and per-vehicle newsvendor budgets, not a 0.5 cut-off and a fixed budget.
- It adapts to the new depot: the failure level, the ranking and the wear tilts are weighted by how well they explain the public scores,
  and the decision hedges over all of them instead of betting on one.
- Its repair-cost model (R² about 0.84 on log cost, out of fold) beats every alternative we tried by a wide margin: ridge, SVR, kNN,
  gradient boosting and all blends with the GP were 16–113% worse on budget loss (`src/v84_costens.py`).

**Limitations.**
- **Which test vehicles fail is the largest open problem, and the available data cannot resolve it.** Rankings from different model
  families disagree strongly on the test depot (CatBoost against the classifier: rank correlation 0.58), yet they fit the public scores
  equally well, and offline validation runs on the training depot. Seven validation designs (plain, wear-weighted, complete cases,
  instrument scenario, cluster hold-out, wear and usage hold-out) were backtested against five changes with known public results; none
  predicted them reliably (`src/v67_valmethods.py`). Our final upload, v82, hedges over these rankings for this reason.
- **The public score is noisy.** It covers about 6,000 vehicles and about 190 failures with heavy-tailed costs; one 8,000 repair moves a
  score by thousands. Under the model a single file's public score has a standard deviation of about 1,850, and all our files scored
  above their expectation by a common amount, consistent with a few expensive failures in the public 30%.
- **The cost model rests on 397 failures**, and its inputs were chosen on them, so some measured gains are selection.
- **The posterior is fitted to the public scores**, so part of its edge may be fitting the public subset rather than the private 70%.

**Leaderboard (criterion a).** Public leaderboard: **rank 13**. Best public score 16,388.51 (Model 3); the final upload, v82 (the Bayes
decision hedged over every ranking family, with Model 3's repair costs), scored 16,440.10, +52 against Model 3 and within the noise.
The private ranking (70% of test) decides the final standing.

## 10. Individual contributions


| Member | GitHub | Contributions |
|---|---|---|
| Vaibhav Sahni | `vaibhav-sahni` | Repository, validation protocol (locked hold-out, fixed folds, exact cost scorer) and submission log; v8 candidate comparisons, versions v9–v31 and v35–v82: depot-shift posterior (v10), robust cost model (v12), level probes, censoring and ranking probes (v13–v24), the public-score **harness** and Bayes decision (v28–v29), and the Gaussian-process refinements leading to **Model 3** (v37, v42, v51) with their ablations; rebuild notebooks, findings log and this report |
| Daksh Sheoran | `Daksh183` | Cost scorer, baselines and v1 sensor analysis (v0–v1); v2 classifier and the **cost-based decision rule of Model 1** (v3) and v3b; v0–v5 notebook builder and histogram discovery scripts; **v32 histogram kernel, the v33 Gaussian-process repair-cost model and Model 2 (v34)**; the GP is the basis of Models 2 and 3 |
| Shivanshu Kumar Singh | `Shivanshu-Kr-Singh` | **Probability calibration** pipeline (v4: Platt and isotonic, `src/calib.py`, notebook kit); **depot-shift correction** (v7: label-shift EM, its simulation check, SHAP and regret analysis; public 19,067); follow-up and missingness variants tested and rejected (v6); feature-drift validation tool (v8) |
| Naveen Jangra | `Naveen23-hub` | **Per-vehicle repair-cost model** (v5) used for budgets and flags; histogram-shape features and their repair-cost validation over 15 folds (v8) |